# Claim Risk Scoring + LangChain Summary Generation

**Goal:** Combine the independent Visual Damage Score (Notebook 2) and Acoustic Stress Score (Notebook 3) into a single, explainable **Claim Risk Score**, then use LangChain to generate a human-readable claim report with a recommendation.

**This is where LangChain enters the pipeline** — everything before this was detection/feature extraction (YOLO, librosa). This notebook is the reasoning/synthesis layer on top.

**Fully offline by default:** uses a local LLM via **Ollama** if available, otherwise falls back to a clean templated summary — either way, no external API calls. The architecture is written so swapping to Claude/GPT via API later is a one-line change (see the Phase 2 note at the end).

## 1. Setup: Load Both Reports

In [1]:
# pip install langchain langchain-community pydantic --quiet
# Optional local LLM: install Ollama separately (https://ollama.com), then run: ollama pull llama3

import json
import os
from pathlib import Path
from typing import List, Optional

VISUAL_REPORTS_PATH = '/kaggle/input/datasets/ronak038/reports/all_videos_visual_reports.json'
AUDIO_REPORTS_PATH = '/kaggle/input/datasets/ronak038/reports/all_videos_audio_reports.json'

with open(VISUAL_REPORTS_PATH) as f:
    visual_reports = json.load(f)

with open(AUDIO_REPORTS_PATH) as f:
    audio_reports = json.load(f)

print(f'Loaded {len(visual_reports)} visual reports and {len(audio_reports)} audio reports.')

# Sanity check: same videos in both?
common_videos = set(visual_reports.keys()) & set(audio_reports.keys())
print(f'{len(common_videos)} videos have both visual and audio reports available.')

Loaded 26 visual reports and 26 audio reports.
26 videos have both visual and audio reports available.


## 2. Merge Into Combined Per-Video Records

In [2]:
combined_records = {}

for video_name in common_videos:
    v = visual_reports[video_name]
    a = audio_reports[video_name]

    combined_records[video_name] = {
        'video_name': video_name,
        'visual_damage_score': v.get('visual_damage_score', 0),
        'damage_types_found': v.get('damage_types_found', []),
        'damage_summary': v.get('damage_summary', []),
        'acoustic_stress_score': a.get('acoustic_stress_score', 0),
        'narration_style_ground_truth': a.get('narration_style_ground_truth'),
        'transcribed_text': a.get('transcribed_text')
    }

print(f'Merged {len(combined_records)} records.')
print(json.dumps(list(combined_records.values())[0], indent=2))

Merged 26 records.
{
  "video_name": "claim_single_tire_flat_v1.mp4",
  "visual_damage_score": 96.3,
  "damage_types_found": [
    "tire flat",
    "dent",
    "scratch",
    "lamp broken"
  ],
  "damage_summary": [
    {
      "damage_type": "tire flat",
      "frames_detected_in": 10,
      "frame_consistency_pct": 55.6,
      "max_confidence": 0.967,
      "avg_confidence": 0.838,
      "best_timestamp_sec": 0.0
    },
    {
      "damage_type": "dent",
      "frames_detected_in": 12,
      "frame_consistency_pct": 66.7,
      "max_confidence": 0.933,
      "avg_confidence": 0.616,
      "best_timestamp_sec": 4.73
    },
    {
      "damage_type": "scratch",
      "frames_detected_in": 8,
      "frame_consistency_pct": 44.4,
      "max_confidence": 0.771,
      "avg_confidence": 0.508,
      "best_timestamp_sec": 1.37
    },
    {
      "damage_type": "lamp broken",
      "frames_detected_in": 6,
      "frame_consistency_pct": 33.3,
      "max_confidence": 0.401,
      "avg_confiden

## 3. Compute the Combined Claim Risk Score

**Weighting rationale (documented, tunable — call this out explicitly in your demo):**
- Visual damage carries more weight (65%) since it's direct physical evidence.
- Acoustic stress carries less weight (35%) since it's a behavioral signal, not proof of anything on its own — it's meant to *flag for review*, not *convict*.
- These weights are a reasonable starting point for a prototype, not a validated actuarial model. Phase 2 would calibrate these against real adjuster-labeled outcomes.

In [4]:
VISUAL_WEIGHT = 0.65
AUDIO_WEIGHT = 0.35


def compute_claim_risk_score(visual_score, audio_score):
    return round(VISUAL_WEIGHT * visual_score + AUDIO_WEIGHT * audio_score, 1)


for record in combined_records.values():
    record['claim_risk_score'] = compute_claim_risk_score(
        record['visual_damage_score'], record['acoustic_stress_score']
    )

## 4. Signal Agreement / Conflict Detection

This is the genuinely useful part that a visual-only or audio-only system would miss. Four scenarios:

| Visual Damage | Acoustic Stress | Interpretation |
|---|---|---|
| High | High | Consistent — severe damage, distressed claimant. Expected pattern, lower scrutiny needed. |
| Low | Low | Consistent — minor damage, calm claimant. Routine, fast-track candidate. |
| High | Low | **Flag** — severe damage described very calmly. Could be innocent (claimant is simply composed), but worth a second look. |
| Low | High | **Flag** — high distress over minor visible damage. Could indicate anxiety unrelated to the claim, exaggeration, or an attempt to justify a bigger payout. |

Thresholds below (40/60 split) are a simple, documented starting point — tune based on your score distributions once you have more real data.

In [5]:
def classify_signal_agreement(visual_score, audio_score, low_threshold=40, high_threshold=60):
    visual_high = visual_score >= high_threshold
    visual_low = visual_score <= low_threshold
    audio_high = audio_score >= high_threshold
    audio_low = audio_score <= low_threshold

    if visual_high and audio_high:
        return 'consistent_high', 'Severe damage with high claimant distress — consistent pattern.'
    elif visual_low and audio_low:
        return 'consistent_low', 'Minor damage with calm claimant — consistent, routine pattern.'
    elif visual_high and audio_low:
        return 'conflict_calm_severe', 'FLAG: Severe visible damage but unusually calm narration — worth a closer look.'
    elif visual_low and audio_high:
        return 'conflict_distressed_minor', 'FLAG: High distress despite minor visible damage — worth a closer look.'
    else:
        return 'ambiguous', 'Mixed/moderate signals — no strong pattern either way.'


for record in combined_records.values():
    category, explanation = classify_signal_agreement(
        record['visual_damage_score'], record['acoustic_stress_score']
    )
    record['signal_agreement_category'] = category
    record['signal_agreement_note'] = explanation

for name, r in combined_records.items():
    print(f"{name}: visual={r['visual_damage_score']}, audio={r['acoustic_stress_score']}, "
          f"risk={r['claim_risk_score']} → {r['signal_agreement_category']}")

claim_single_tire_flat_v1.mp4: visual=96.3, audio=65.2, risk=85.4 → consistent_high
claim_edge_long_walkaround.mp4: visual=100.0, audio=42.5, risk=79.9 → ambiguous
claim_mixed_4.mp4: visual=81.3, audio=36.3, risk=65.5 → conflict_calm_severe
claim_single_crack_v1.mp4: visual=100.0, audio=61.1, risk=86.4 → consistent_high
claim_mixed_2.mp4: visual=100.0, audio=61.1, risk=86.4 → consistent_high
claim_single_lamp_broken_v2.mp4: visual=86.3, audio=52.6, risk=74.5 → ambiguous
claim_single_tire_flat_v2.mp4: visual=87.8, audio=42.1, risk=71.8 → ambiguous
claim_single_lamp_broken_v1.mp4: visual=80.7, audio=45.9, risk=68.5 → ambiguous
claim_single_glass_shatter_v2.mp4: visual=97.0, audio=56.3, risk=82.8 → ambiguous
claim_pair_dent_tire_flat.mp4: visual=100.0, audio=61.0, risk=86.3 → consistent_high
claim_pair_dent_scratch.mp4: visual=96.1, audio=61.5, risk=84.0 → consistent_high
claim_mixed_3.mp4: visual=80.2, audio=39.8, risk=66.1 → conflict_calm_severe
claim_mixed_6.mp4: visual=70.1, audio=31.

## 5. LangChain Setup: Local LLM with Templated Fallback

Tries Ollama first (fully offline once a model is pulled). If Ollama isn't installed/running, falls back to a clean templated summary — so this notebook still works even without any LLM setup, which matters if you're short on time today.

In [7]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.

In [10]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 193 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (374 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 121026 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [11]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [12]:
import subprocess
import os
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [13]:
!curl http://localhost:11434/api/tags

{"models":[]}

In [14]:
!ollama pull llama3

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏  22 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  56 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   3% ▕                  ▏ 123 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   4% ▕                  ▏ 194 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   5% ▕                  ▏ 234 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   7% ▕█                 ▏ 316 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   9% ▕█                 ▏ 396 MB/4.7 GB                  pulling m

In [15]:
!ollama list

]11;?\NAME             ID              SIZE      MODIFIED       
llama3:latest    365c0bd3c000    4.7 GB    11 seconds ago    


In [17]:
!pip install -U langchain-ollama

In [18]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3")

response = llm.invoke('Say "ready" and nothing else.')
print(response)

Ready


In [19]:
USE_LOCAL_LLM = True  # set False to force the templated fallback regardless of Ollama availability

llm = None
if USE_LOCAL_LLM:
    try:
        from langchain_community.llms import Ollama
        llm = Ollama(model='llama3')
        _ = llm.invoke('Say "ready" and nothing else.')  # quick smoke test
        print('Local Ollama LLM connected successfully — using LangChain + local LLM for summaries.')
    except Exception as e:
        print(f'Could not connect to local Ollama LLM ({type(e).__name__}: {e})')
        print('Falling back to templated summary generation. To use a real local LLM:')
        print('  1. Install Ollama: https://ollama.com')
        print('  2. Run: ollama pull llama3')
        print('  3. Re-run this cell.')
        llm = None

Local Ollama LLM connected successfully — using LangChain + local LLM for summaries.


## 6. Structured Output Schema

Using Pydantic to define the exact JSON shape we want back — this makes the output directly usable by downstream systems (e.g., a claims database or dashboard), not just readable prose. This works whether the content comes from a real LLM or the templated fallback, since both populate the same schema.

In [20]:
from pydantic import BaseModel, Field


class ClaimReport(BaseModel):
    claim_risk_score: float = Field(description='Combined 0-100 risk score')
    risk_level: str = Field(description='One of: Low, Medium, High')
    key_findings: List[str] = Field(description='Bullet list of the most important observations')
    signal_agreement: str = Field(description='Whether visual and audio signals agree or conflict, and why')
    recommendation: str = Field(description='Recommended next action for the claims adjuster')
    summary: str = Field(description='2-3 sentence natural language summary of the claim')


def risk_level_from_score(score):
    if score >= 65:
        return 'High'
    elif score >= 35:
        return 'Medium'
    else:
        return 'Low'

## 7. Prompt Template (used when a local LLM is available)

Feeds the structured detection data into the LLM with clear instructions to reason over it — the LLM's job is synthesis/explanation, not detection (that already happened in Notebooks 2 & 3).

In [22]:
from langchain_core.prompts import PromptTemplate

CLAIM_SUMMARY_PROMPT = PromptTemplate(
    input_variables=['video_name', 'damage_types', 'damage_details', 'visual_score',
                      'audio_score', 'risk_score', 'agreement_note'],
    template="""You are assisting an insurance claims adjuster by summarizing an automated claim video analysis.
Do not invent details beyond what is provided below.

Claim video: {video_name}
Detected damage types: {damage_types}
Damage detail: {damage_details}
Visual Damage Score (0-100): {visual_score}
Acoustic Stress Score (0-100): {audio_score}
Combined Claim Risk Score (0-100): {risk_score}
Signal agreement note: {agreement_note}

Write a concise 2-3 sentence summary of this claim for the adjuster, in plain professional language.
Then give ONE clear recommendation for next steps (e.g., fast-track approval, standard review, or manual investigation).
Be factual and measured — do not accuse the claimant of fraud; only note where signals suggest closer review is warranted.

Summary:"""
)

## 8. Generate the Report (LLM Path or Templated Fallback)

In [23]:
def generate_summary_with_llm(record):
    damage_details_str = '; '.join(
        f"{d['damage_type']} (confidence {d['max_confidence']}, seen in {d['frame_consistency_pct']}% of frames)"
        for d in record['damage_summary']
    ) or 'No damage detected'

    prompt_text = CLAIM_SUMMARY_PROMPT.format(
        video_name=record['video_name'],
        damage_types=', '.join(record['damage_types_found']) or 'none',
        damage_details=damage_details_str,
        visual_score=record['visual_damage_score'],
        audio_score=record['acoustic_stress_score'],
        risk_score=record['claim_risk_score'],
        agreement_note=record['signal_agreement_note']
    )

    response_text = llm.invoke(prompt_text)
    return response_text.strip()


def generate_summary_templated(record):
    """Clean, deterministic fallback — no LLM required. Good enough for a working demo
    if local LLM setup isn't feasible today; swap for the LLM path when time allows.
    """
    damage_types = record['damage_types_found']
    risk_level = risk_level_from_score(record['claim_risk_score'])

    if damage_types:
        top_damage = max(record['damage_summary'], key=lambda d: d['max_confidence'])
        damage_phrase = (f"{', '.join(damage_types)} detected, most notably {top_damage['damage_type']} "
                          f"at {top_damage['max_confidence']*100:.0f}% confidence")
    else:
        damage_phrase = 'no significant damage detected'

    summary = (f"This claim shows {damage_phrase}. "
               f"Acoustic analysis of the claimant narration produced a stress score of "
               f"{record['acoustic_stress_score']}/100. {record['signal_agreement_note']}")

    recommendation_map = {
        'consistent_high': 'Proceed with standard claims processing; evidence is internally consistent.',
        'consistent_low': 'Fast-track for approval; low damage, low risk indicators.',
        'conflict_calm_severe': 'Route to manual adjuster review due to signal mismatch (severe damage, calm narration).',
        'conflict_distressed_minor': 'Route to manual adjuster review due to signal mismatch (high distress, minor damage).',
        'ambiguous': 'Standard review recommended; no strong signal in either direction.'
    }
    recommendation = recommendation_map.get(record['signal_agreement_category'], 'Standard review recommended.')

    return {
        'summary': summary,
        'recommendation': recommendation,
        'risk_level': risk_level
    }


def build_claim_report(record):
    if llm is not None:
        try:
            llm_text = generate_summary_with_llm(record)
            summary_text = llm_text
            recommendation_text = None
        except Exception as e:
            print(f"LLM generation failed for {record['video_name']} ({e}), using templated fallback.")
            fallback = generate_summary_templated(record)
            summary_text = fallback['summary']
            recommendation_text = fallback['recommendation']
    else:
        fallback = generate_summary_templated(record)
        summary_text = fallback['summary']
        recommendation_text = fallback['recommendation']

    report = ClaimReport(
        claim_risk_score=record['claim_risk_score'],
        risk_level=risk_level_from_score(record['claim_risk_score']),
        key_findings=[
            f"Visual Damage Score: {record['visual_damage_score']}/100",
            f"Damage types detected: {', '.join(record['damage_types_found']) or 'none'}",
            f"Acoustic Stress Score: {record['acoustic_stress_score']}/100"
        ],
        signal_agreement=record['signal_agreement_note'],
        recommendation=recommendation_text or 'See summary.',
        summary=summary_text
    )
    return report

In [24]:
final_reports = {}

for video_name, record in combined_records.items():
    print(f'Generating report: {video_name}...')
    report = build_claim_report(record)
    final_reports[video_name] = report.dict()

print(f'\nGenerated {len(final_reports)} claim reports.')

Generating report: claim_single_tire_flat_v1.mp4...


/tmp/ipykernel_58/170438246.py:6: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  final_reports[video_name] = report.dict()


Generating report: claim_edge_long_walkaround.mp4...


/tmp/ipykernel_58/170438246.py:6: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  final_reports[video_name] = report.dict()


Generating report: claim_mixed_4.mp4...


...

/tmp/ipykernel_58/170438246.py:6: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  final_reports[video_name] = report.dict()


Generating report: claim_single_dent_v2.mp4...


/tmp/ipykernel_58/170438246.py:6: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  final_reports[video_name] = report.dict()


Generating report: claim_pair_crack_glass_shatter.mp4...

Generated 26 claim reports.


/tmp/ipykernel_58/170438246.py:6: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  final_reports[video_name] = report.dict()


## 9. Review the Reports

In [25]:
for video_name, report in final_reports.items():
    print(f"\n{'='*70}")
    print(f"CLAIM REPORT: {video_name}")
    print(f"{'='*70}")
    print(f"Risk Score: {report['claim_risk_score']}/100  ({report['risk_level']})")
    print(f"\nKey Findings:")
    for finding in report['key_findings']:
        print(f"  • {finding}")
    print(f"\nSignal Agreement: {report['signal_agreement']}")
    print(f"\nSummary: {report['summary']}")
    print(f"\nRecommendation: {report['recommendation']}")


CLAIM REPORT: claim_single_tire_flat_v1.mp4
Risk Score: 85.4/100  (High)

Key Findings:
  • Visual Damage Score: 96.3/100
  • Damage types detected: tire flat, dent, scratch, lamp broken
  • Acoustic Stress Score: 65.2/100

Signal Agreement: Severe damage with high claimant distress — consistent pattern.

Summary: Here is a concise summary of the claim:

The claim video analysis indicates that the vehicle sustained significant damage, including a flat tire, dent, and scratch, with a high confidence level in the detection of these damages. The Visual Damage Score and Combined Claim Risk Score both indicate a high level of damage, suggesting a severe incident. The Acoustic Stress Score suggests that the claimant experienced moderate distress during the incident.

Recommendation: Given the high level of damage and the claimant's distress, I recommend that the adjuster fast-track the approval process, with a thorough review of the claim to ensure all damages are properly documented and co

## 10. Save Final Reports

In [26]:
os.makedirs('reports', exist_ok=True)
with open('reports/final_claim_reports.json', 'w') as f:
    json.dump(final_reports, f, indent=2)

print('Saved to reports/final_claim_reports.json')
print('\nThese are your end-to-end demo outputs — pick 2-3 (one high-risk, one low-risk,')
print('one conflicting-signal case) as your key screenshots/talking points for the demo.')

Saved to reports/final_claim_reports.json

These are your end-to-end demo outputs — pick 2-3 (one high-risk, one low-risk,
one conflicting-signal case) as your key screenshots/talking points for the demo.


## Phase 2 Note: Swapping to a Cloud LLM API

Because LangChain abstracts the LLM behind a common interface, moving from local Ollama to a cloud API (e.g., Claude or GPT) later requires changing only Section 5 — swap:

```python
from langchain_community.llms import Ollama
llm = Ollama(model='llama3')
```

for something like:

```python
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model='claude-...', api_key='...')
```

Everything downstream (prompt template, structured output schema, report generation) stays identical. This is exactly the "seamless integration once API access is available" story for your architecture doc.

## What's Next

With this notebook done, the core pipeline is complete end-to-end: video/audio in → detection → scoring → LangChain-generated claim report out. Remaining work:
1. (Optional) Consolidate Notebooks 2, 3, and 5 into a single `run_pipeline(video_path)` function for a cleaner demo.
2. Build a lightweight Streamlit front-end wrapping that pipeline.
3. Write the architecture/README doc (current-state offline design + future-state API roadmap).